## Pacotes e dependências

In [4]:
from urllib import request
import pandas as pd
import numpy as np

### 1. Download do Dataset

In [5]:
url_dados = "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/intercambio_nacional_ho/"
arquivo_dados = "INTERCAMBIO_NACIONAL_"
historico_dados = ["2023", "2024", "2025"]

for ano in historico_dados:
    request.urlretrieve(url_dados+arquivo_dados+ano+".csv", arquivo_dados+ano+".csv")


### 2. Exportar arquivo empilhado

In [8]:
dataframe_final = pd.DataFrame()
for ano in historico_dados:
    dataframe = pd.read_csv(arquivo_dados+ano+".csv", sep=";", encoding="utf-8")
    dataframe_final = pd.concat([dataframe_final, dataframe], ignore_index=True)

dataframe_final.to_csv("INTERCAMBIO_NACIONAL_2023-2025.csv", sep=";", index=False, encoding="utf-8")


## Trigem de dados

In [7]:
# Excluir colunas "id_subsistema_origem" e "id_subsistema_destino"
dataframe_final = dataframe_final.drop(columns=["id_subsistema_origem", "id_subsistema_destino"], errors="ignore")

# Substituir valores de duas colunas SUDESTE -> SUDESTE/CENTRO-OESTE
dataframe_final['nom_subsistema_origem'] = dataframe_final['nom_subsistema_origem'].replace({'SUDESTE': 'SUDESTE/CENTRO-OESTE'})
dataframe_final['nom_subsistema_destino'] = dataframe_final['nom_subsistema_destino'].replace({'SUDESTE': 'SUDESTE/CENTRO-OESTE'})

dataframe_final

,din_instante,nom_subsistema_origem,nom_subsistema_destino,val_intercambiomwmed
0,2023-01-01 00:00:00,NORTE,NORDESTE,-3393.396
1,2023-01-01 00:00:00,NORTE,SUDESTE/CENTRO-OESTE,6351.214
2,2023-01-01 00:00:00,NORDESTE,SUDESTE/CENTRO-OESTE,4009.116
3,2023-01-01 00:00:00,SUDESTE/CENTRO-OESTE,SUL,6933.356
4,2023-01-01 01:00:00,NORTE,NORDESTE,-2956.452
...,...,...,...,...
93499,2025-08-31 22:00:00,SUDESTE/CENTRO-OESTE,SUL,-2328.884
93500,2025-08-31 23:00:00,NORTE,NORDESTE,-5152.965
93501,2025-08-31 23:00:00,NORTE,SUDESTE/CENTRO-OESTE,1043.700
93502,2025-08-31 23:00:00,NORDESTE,SUDESTE/CENTRO-OESTE,6471.169


## Criação de features de calendário

In [ ]:
def add_calendar_features(df):
    """
    Adiciona features de calendário ao DataFrame.
    
    Args:
        df: DataFrame com índice datetime
        
    Returns:
        DataFrame com features de calendário adicionadas
    """
    df = df.copy()
    
    # Features básicas de tempo
    df['hour'] = df.index.hour
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month
    df['day'] = df.index.day
    df['year'] = df.index.year
    
    # Encoding cíclico
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
    df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)
    
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    
    # Identificadores de períodos especiais
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    
    # Simplificação: como não temos uma lista de feriados,
    # vamos apenas fingir que os fins de semana são feriados
    df['is_holiday'] = df['is_weekend']
    
    # Lags importantes (t-24 e t-168)
    df['MW_lag24'] = df['MW'].shift(24)
    df['MW_lag168'] = df['MW'].shift(168)
    
    return df

# Aplicar features de calendário
ts_df_features = add_calendar_features(ts_df)
print("DataFrame com features de calendário:")
print(ts_df_features.head())

In [ ]:
dataframe_final = pd.DataFrame()
for ano in historico_dados:
    dataframe = pd.read_csv(arquivo_dados+ano+".csv", sep=";", encoding="utf-8")
    dataframe_final = pd.concat([dataframe_final, dataframe], ignore_index=True)

if dataframe_final.empty:
    print("DataFrame vazio")

else:
    print()
    print()
